# ComfyUI Workflow Batch Testing

Run batch tests on all workflows in the Comfy_Workflow dataset.


In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

from agents.comfy import BatchTestRunner
import pandas as pd


In [ ]:
# Load test configuration
config_path = project_root / "Datasets" / "Comfy_Workflow" / "test_config.json"
runner = BatchTestRunner()
test_cases = runner.load_config(config_path)

print(f"Loaded {len(test_cases)} test cases")


In [ ]:
# Run batch tests
summary = runner.run_batch(test_cases, max_workers=4, resume=True)

print(f"\n{'='*60}")
print(f"Test Results Summary")
print(f"{'='*60}")
print(f"Total: {summary['total']} | Passed: {summary['passed']} | Failed: {summary['failed']} | Elapsed: {summary['elapsed_seconds']:.1f}s")


In [ ]:
# Display results table
results_data = []
for test_id, result in summary['results'].items():
    results_data.append({
        'Test ID': test_id,
        'Status': result.get('status', 'UNKNOWN'),
        'Elapsed (s)': f"{result.get('elapsed', 0):.1f}",
        'Error': result.get('error', '')[:50] if result.get('error') else ''
    })

df = pd.DataFrame(results_data)
print(df.to_string(index=False))


In [ ]:
# Show failed tests
failed = [r for r in summary['results'].values() if r.get('status') == 'FAIL']
if failed:
    print(f"\n{'='*60}")
    print(f"Failed Tests ({len(failed)}):")
    print(f"{'='*60}")
    for result in failed:
        print(f"- {result.get('test_id')}: {result.get('error', 'Unknown error')}")
else:
    print("\nAll tests passed!")
